# Visualizacion y metricas de segmentacion

Este notebook arranca desde los archivos `.czi.labeling` generados a mano. Esos archivos no guardan una imagen de mascara, sino listas de coordenadas `[x, y]` por etiqueta. Aca reconstruimos una mascara binaria: `0 = fondo`, `1 = amastigote`.

Que la label manual sea binaria no es un problema para evaluar. Para tu objetivo, la metrica principal va a ser deteccion/conteo: cuantos amastigotes manuales fueron detectados y cuantos objetos predijo el programa. Las metricas de area (`IoU`, `Dice`) quedan como control secundario, no como criterio central.

In [ ]:
from pathlib import Path
import json

from PIL import Image, ImageDraw
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "amastigotes DAPI etiquetados").exists() and (PROJECT_ROOT.parent / "amastigotes DAPI etiquetados").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
LABEL_DIR = PROJECT_ROOT / "amastigotes DAPI etiquetados"
DAPI_DIR = PROJECT_ROOT / "dapi"

# Cuando tengas resultados del programa, apunta esto a la carpeta que contiene
# job_.../images/.../parasite_mask.tiff
PRED_ROOT = PROJECT_ROOT / "process"

print(PROJECT_ROOT)


In [ ]:
def available_label_ids(label_dir=LABEL_DIR):
    ids = []
    for path in sorted(Path(label_dir).glob("*.labeling")):
        name = path.name
        if name.endswith(".czi.labeling"):
            ids.append(name.removesuffix(".czi.labeling"))
        else:
            ids.append(path.stem)
    return ids


def labeling_path_for(image_id, label_dir=LABEL_DIR):
    label_dir = Path(label_dir)
    direct = label_dir / f"{image_id}.czi.labeling"
    if direct.exists():
        return direct

    hits = sorted(label_dir.glob(f"{image_id}*.labeling"))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"No encontre labeling para {image_id} en {label_dir}")


def dapi_path_for(image_id, dapi_dir=DAPI_DIR):
    dapi_dir = Path(dapi_dir)
    for suffix in (".tiff", ".tif", ".png", ".jpg", ".jpeg"):
        candidate = dapi_dir / f"{image_id}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No encontre imagen DAPI para {image_id} en {dapi_dir}")


ids = available_label_ids()
print(f"Labels encontradas: {len(ids)}")
print(ids[:12])


## Reconstruccion de una label manual

El `.labeling` tiene coordenadas en formato `[x, y]`. Para dibujar la mascara, cada punto se pinta en una imagen `L` de Pillow con tamano tomado de `interval.max` e `interval.min`.

In [ ]:
BACKGROUND_LABELS = {"background"}


def load_labeling_mask_pil(labeling_path):
    labeling_path = Path(labeling_path)
    data = json.loads(labeling_path.read_text(encoding="utf-8"))

    min_x, min_y = data["interval"]["min"]
    max_x, max_y = data["interval"]["max"]
    width = int(max_x - min_x + 1)
    height = int(max_y - min_y + 1)

    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)

    counts = {}
    for label_name, coords in data.get("labels", {}).items():
        if label_name.strip().lower() in BACKGROUND_LABELS:
            continue
        if not coords:
            counts[label_name] = 0
            continue

        points = [(int(x) - min_x, int(y) - min_y) for x, y in coords]
        draw.point(points, fill=255)
        counts[label_name] = len(points)

    return mask, data, counts


def image_to_8bit_preview(image):
    """Convierte una imagen 2D a preview 8-bit usando el rango real de intensidades."""
    image_i = image.convert("I")
    lo, hi = image_i.getextrema()
    if hi <= lo:
        return Image.new("L", image_i.size, 0)

    scale = 255.0 / max(hi - lo, 1)
    return image_i.point(lambda p: (p - lo) * scale).convert("L")


def load_dapi_preview(image_id):
    image_path = dapi_path_for(image_id)
    image = Image.open(image_path)
    return image_to_8bit_preview(image)


def overlay_mask(base_l, mask_l, color=(0, 255, 74), alpha=135):
    base_rgba = base_l.convert("RGBA")
    color_layer = Image.new("RGBA", base_rgba.size, (*color, 0))
    alpha_mask = mask_l.point(lambda p: alpha if p else 0)
    color_layer.putalpha(alpha_mask)
    return Image.alpha_composite(base_rgba, color_layer).convert("RGB")


def resize_to_width(image, width):
    if image.width == width:
        return image
    height = round(image.height * width / image.width)
    return image.resize((width, height))


def preview_grid(image_id, panel_width=620):
    label_path = labeling_path_for(image_id)
    mask, data, counts = load_labeling_mask_pil(label_path)
    dapi = load_dapi_preview(image_id)

    if dapi.size != mask.size:
        raise ValueError(f"Tamano distinto: DAPI {dapi.size}, label {mask.size}")

    overlay = overlay_mask(dapi, mask)
    mask_rgb = Image.new("RGB", mask.size, (0, 0, 0))
    mask_color = Image.new("RGB", mask.size, (0, 255, 74))
    mask_rgb.paste(mask_color, mask=mask)

    panels = [
        ("DAPI", dapi.convert("RGB")),
        ("Label manual", mask_rgb),
        ("Overlay", overlay),
    ]
    panels = [(title, resize_to_width(img, panel_width)) for title, img in panels]

    title_h = 28
    gap = 12
    sheet_w = panel_width * len(panels) + gap * (len(panels) - 1)
    sheet_h = title_h + max(img.height for _, img in panels)
    sheet = Image.new("RGB", (sheet_w, sheet_h), (20, 20, 20))
    draw = ImageDraw.Draw(sheet)

    x = 0
    for title, img in panels:
        draw.text((x + 6, 6), title, fill=(235, 235, 235))
        sheet.paste(img, (x, title_h))
        x += panel_width + gap

    total_pixels = sum(counts.values())
    print(f"{image_id}: {mask.size[0]} x {mask.size[1]} px | pixeles label={total_pixels} | labels={counts}")
    return sheet


In [ ]:
IMAGE_ID = "Snap-12879"
display(preview_grid(IMAGE_ID, panel_width=520))


## Guardar mascaras reconstruidas como TIFF/PNG

Esto es util si queres inspeccionarlas fuera del notebook o usarlas luego en un pipeline de evaluacion. Por defecto no corre para todas; cambia `EXPORT_ALL = True` si queres exportarlas.

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "evaluacion_labels_reconstruidas"
EXPORT_ALL = False

if EXPORT_ALL:
    EXPORT_DIR.mkdir(exist_ok=True)
    for image_id in available_label_ids():
        mask, _, _ = load_labeling_mask_pil(labeling_path_for(image_id))
        mask.save(EXPORT_DIR / f"{image_id}_gt_mask.png")
    print(f"Listo: {EXPORT_DIR}")
else:
    print("Export desactivado. Cambia EXPORT_ALL = True para guardar todas las mascaras.")


## Metricas contra las predicciones del programa

La salida del programa para amastigotes es `parasite_mask.tiff`, con `0 = fondo` y `1..n = instancias`. La evaluacion principal matchea objetos de forma relajada: si una prediccion cae sobre/cerca de una label manual, cuenta como detectada aunque el area no sea exactamente igual. Tambien se calculan `IoU` y `Dice`, pero solo como diagnostico de forma.

In [ ]:
try:
    import numpy as np
    import pandas as pd
    import tifffile
    from scipy import ndimage as ndi

    HAS_NUMPY_STACK = True
except ModuleNotFoundError as exc:
    HAS_NUMPY_STACK = False
    print("Faltan dependencias para metricas numericas:", exc)
    print("La visualizacion anterior funciona igual. Para metricas instala numpy, pandas, tifffile y scipy en el kernel.")


In [ ]:
if HAS_NUMPY_STACK:

    def load_labeling_mask_np(image_id):
        mask_pil, _, _ = load_labeling_mask_pil(labeling_path_for(image_id))
        return np.asarray(mask_pil) > 0


    def read_prediction_mask(mask_path):
        arr = tifffile.imread(str(mask_path))
        arr = np.squeeze(arr)
        while arr.ndim > 2:
            arr = arr[0]
        return arr


    def pixel_metrics(gt_mask, pred_mask):
        gt = np.asarray(gt_mask, dtype=bool)
        pred = np.asarray(pred_mask) > 0
        if gt.shape != pred.shape:
            raise ValueError(f"Shapes distintas: gt={gt.shape}, pred={pred.shape}")

        tp = int(np.logical_and(gt, pred).sum())
        fp = int(np.logical_and(~gt, pred).sum())
        fn = int(np.logical_and(gt, ~pred).sum())
        tn = int(np.logical_and(~gt, ~pred).sum())

        return {
            "tp_px": tp,
            "fp_px": fp,
            "fn_px": fn,
            "tn_px": tn,
            "iou_px": tp / (tp + fp + fn) if (tp + fp + fn) else 1.0,
            "dice_px": (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 1.0,
            "precision_px": tp / (tp + fp) if (tp + fp) else 0.0,
            "recall_px": tp / (tp + fn) if (tp + fn) else 0.0,
        }


    def _as_instance_labels(mask_or_labels):
        arr = np.asarray(mask_or_labels)
        if arr.max(initial=0) <= 1:
            lab, _ = ndi.label(arr > 0)
            return lab.astype(np.int32, copy=False)
        return arr.astype(np.int32, copy=False)


    def object_metrics(gt_mask, pred_mask, iou_threshold=0.5):
        gt_lab = _as_instance_labels(gt_mask)
        pred_lab = _as_instance_labels(pred_mask)
        if gt_lab.shape != pred_lab.shape:
            raise ValueError(f"Shapes distintas: gt={gt_lab.shape}, pred={pred_lab.shape}")

        n_gt = int(gt_lab.max(initial=0))
        n_pred = int(pred_lab.max(initial=0))
        if n_gt == 0 and n_pred == 0:
            return {
                "gt_objects": 0,
                "pred_objects": 0,
                "matched_objects": 0,
                "precision_obj": 1.0,
                "recall_obj": 1.0,
                "f1_obj": 1.0,
                "mean_iou_matches": 1.0,
            }

        gt_area = np.bincount(gt_lab.ravel(), minlength=n_gt + 1)
        pred_area = np.bincount(pred_lab.ravel(), minlength=n_pred + 1)

        flat_gt = gt_lab.ravel().astype(np.int64)
        flat_pred = pred_lab.ravel().astype(np.int64)
        both = (flat_gt > 0) & (flat_pred > 0)
        pair_key = flat_gt[both] * (n_pred + 1) + flat_pred[both]
        overlaps = np.bincount(pair_key, minlength=(n_gt + 1) * (n_pred + 1))

        candidates = []
        for key in np.nonzero(overlaps)[0]:
            gid = int(key // (n_pred + 1))
            pid = int(key % (n_pred + 1))
            if gid == 0 or pid == 0:
                continue
            inter = int(overlaps[key])
            union = int(gt_area[gid] + pred_area[pid] - inter)
            iou = inter / union if union else 0.0
            candidates.append((iou, gid, pid))

        candidates.sort(reverse=True)
        used_gt = set()
        used_pred = set()
        matches = []
        for iou, gid, pid in candidates:
            if iou < iou_threshold:
                break
            if gid in used_gt or pid in used_pred:
                continue
            used_gt.add(gid)
            used_pred.add(pid)
            matches.append((gid, pid, iou))

        tp = len(matches)
        precision = tp / n_pred if n_pred else 0.0
        recall = tp / n_gt if n_gt else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

        return {
            "gt_objects": n_gt,
            "pred_objects": n_pred,
            "matched_objects": tp,
            "false_positive_objects": max(n_pred - tp, 0),
            "false_negative_objects": max(n_gt - tp, 0),
            "precision_obj": precision,
            "recall_obj": recall,
            "f1_obj": f1,
            "mean_iou_matches": float(np.mean([m[2] for m in matches])) if matches else 0.0,
        }


    def _dilate_instance_labels(labels, radius_px):
        if radius_px <= 0:
            return labels

        out = np.zeros_like(labels, dtype=np.int32)
        structure = np.ones((2 * radius_px + 1, 2 * radius_px + 1), dtype=bool)
        for obj_id in range(1, int(labels.max(initial=0)) + 1):
            expanded = ndi.binary_dilation(labels == obj_id, structure=structure)
            out[(out == 0) & expanded] = obj_id
        return out


    def detection_count_metrics(gt_mask, pred_mask, dilation_radius_px=3, min_overlap_pixels=1):
        """Metricas pensadas para deteccion/conteo, no para igualdad exacta de area."""
        gt_lab = _as_instance_labels(gt_mask)
        pred_lab = _as_instance_labels(pred_mask)
        if gt_lab.shape != pred_lab.shape:
            raise ValueError(f"Shapes distintas: gt={gt_lab.shape}, pred={pred_lab.shape}")

        n_gt = int(gt_lab.max(initial=0))
        n_pred = int(pred_lab.max(initial=0))
        count_error = n_pred - n_gt
        abs_count_error = abs(count_error)
        count_accuracy = max(0.0, 1.0 - (abs_count_error / n_gt)) if n_gt else (1.0 if n_pred == 0 else 0.0)

        if n_gt == 0 or n_pred == 0:
            matched = 0
            precision = 1.0 if n_pred == 0 else 0.0
            recall = 1.0 if n_gt == 0 else 0.0
            f1 = 1.0 if n_gt == 0 and n_pred == 0 else 0.0
            return {
                "gt_count": n_gt,
                "pred_count": n_pred,
                "count_error": count_error,
                "abs_count_error": abs_count_error,
                "count_accuracy": count_accuracy,
                "matched_detections": matched,
                "missed_detections": n_gt,
                "extra_detections": n_pred,
                "precision_det": precision,
                "recall_det": recall,
                "f1_det": f1,
            }

        gt_match_lab = _dilate_instance_labels(gt_lab, dilation_radius_px)
        flat_gt = gt_match_lab.ravel().astype(np.int64)
        flat_pred = pred_lab.ravel().astype(np.int64)
        both = (flat_gt > 0) & (flat_pred > 0)
        pair_key = flat_gt[both] * (n_pred + 1) + flat_pred[both]
        overlaps = np.bincount(pair_key, minlength=(n_gt + 1) * (n_pred + 1))

        candidates = []
        for key in np.nonzero(overlaps)[0]:
            gid = int(key // (n_pred + 1))
            pid = int(key % (n_pred + 1))
            if gid == 0 or pid == 0:
                continue
            overlap = int(overlaps[key])
            if overlap >= min_overlap_pixels:
                candidates.append((overlap, gid, pid))

        candidates.sort(reverse=True)
        used_gt = set()
        used_pred = set()
        for overlap, gid, pid in candidates:
            if gid in used_gt or pid in used_pred:
                continue
            used_gt.add(gid)
            used_pred.add(pid)

        matched = len(used_gt)
        precision = matched / n_pred if n_pred else 0.0
        recall = matched / n_gt if n_gt else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

        return {
            "gt_count": n_gt,
            "pred_count": n_pred,
            "count_error": count_error,
            "abs_count_error": abs_count_error,
            "count_accuracy": count_accuracy,
            "matched_detections": matched,
            "missed_detections": max(n_gt - matched, 0),
            "extra_detections": max(n_pred - matched, 0),
            "precision_det": precision,
            "recall_det": recall,
            "f1_det": f1,
        }


    def evaluate_one(image_id, pred_mask_path, iou_threshold=0.5, detection_radius_px=3):
        gt = load_labeling_mask_np(image_id)
        pred = read_prediction_mask(pred_mask_path)
        return {
            "image_id": image_id,
            "pred_mask": str(pred_mask_path),
            **detection_count_metrics(gt, pred, dilation_radius_px=detection_radius_px),
            **pixel_metrics(gt, pred),
            **object_metrics(gt, pred, iou_threshold=iou_threshold),
        }

else:
    print("Salteando definicion de metricas numericas porque faltan dependencias.")


### Evaluar una mascara puntual

Ajusta `PRED_MASK_PATH` a un archivo real `parasite_mask.tiff` generado por el programa. La carpeta suele tener forma `.../job_<id>/images/0001__Snap-12879/parasite_mask.tiff`.

In [ ]:
PRED_MASK_PATH = PRED_ROOT / "job_id" / "job_job_id" / "images" / "0001__Snap-12879" / "parasite_mask.tiff"

if HAS_NUMPY_STACK and PRED_MASK_PATH.exists():
    display(pd.DataFrame([evaluate_one("Snap-12879", PRED_MASK_PATH, detection_radius_px=3)]))
else:
    print("Ajusta PRED_MASK_PATH a un parasite_mask.tiff real para evaluar una imagen.")


### Evaluar una carpeta completa de resultados

Esto busca recursivamente todos los `parasite_mask.tiff` bajo `PRED_ROOT`. El `image_id` se toma del nombre de carpeta: `0001__Snap-12879` -> `Snap-12879`.

In [ ]:
if HAS_NUMPY_STACK:

    def image_id_from_prediction_path(mask_path):
        folder_name = Path(mask_path).parent.name
        if "__" in folder_name:
            return folder_name.split("__", 1)[1]
        return Path(mask_path).stem


    def evaluate_prediction_tree(pred_root=PRED_ROOT, iou_threshold=0.5, detection_radius_px=3):
        rows = []
        missing_labels = []
        for pred_path in sorted(Path(pred_root).rglob("parasite_mask.tiff")):
            image_id = image_id_from_prediction_path(pred_path)
            try:
                labeling_path_for(image_id)
            except FileNotFoundError:
                missing_labels.append(image_id)
                continue
            rows.append(evaluate_one(image_id, pred_path, iou_threshold=iou_threshold, detection_radius_px=detection_radius_px))

        df = pd.DataFrame(rows)
        if missing_labels:
            print("Predicciones sin label manual:", sorted(set(missing_labels))[:20])
        return df


    df_eval = evaluate_prediction_tree(PRED_ROOT, iou_threshold=0.5, detection_radius_px=3)
    print(f"Imagenes evaluadas: {len(df_eval)}")
    display(df_eval)
else:
    print("No se puede evaluar la carpeta hasta instalar las dependencias numericas.")
